<a href="https://colab.research.google.com/github/XULULUCHENAN12/ChatGPT-Linebot-using-python-flask-on-vercel/blob/master/GPT-SoVITS-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

👇 Just Click the Play Button (Run) and Wait for Public Gradio Link

In [1]:
# @title Do Everything For Me and Launch Gradio
%cd /content
!git clone https://github.com/RVC-Boss/GPT-SoVITS
%cd /content/GPT-SoVITS

!conda install -c conda-forge gcc gxx ffmpeg cmake
!conda install pytorch==2.1.1 torchvision==0.16.1 torchaudio==2.1.1 pytorch-cuda=11.8 -c pytorch -c nvidia

!pip install pyyaml  # This addresses the ModuleNotFoundError for yaml
!pip install numba==0.56.4  # Trying to install numba separately to see if it resolves the issue

!pip install -r requirements.txt

!python webui.py

/content
Cloning into 'GPT-SoVITS'...
remote: Enumerating objects: 5930, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 5930 (delta 52), reused 16 (delta 16), pack-reused 5844 (from 3)
Receiving objects: 100% (5930/5930), 15.13 MiB | 14.59 MiB/s, done.
Resolving deltas: 100% (3382/3382), done.
/content/GPT-SoVITS
/bin/bash: line 1: conda: command not found
/bin/bash: line 1: conda: command not found
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.8 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not

In [1]:
!sed -i '/^opencc/d' /content/GPT-SoVITS/requirements.txt
!pip install opencc-python-reimplemented
!pip install -r /content/GPT-SoVITS/requirements.txt
%cd /content/GPT-SoVITS
!python webui.py

Ignoring onnxruntime: markers 'platform_machine == "aarch64" or platform_machine == "arm64"' don't match your environment
/content/GPT-SoVITS
    GPT_SoVITS/pretrained_models/v2Pro/s2Gv2Pro.pth
    GPT_SoVITS/pretrained_models/v2Pro/s2Dv2Pro.pth
    GPT_SoVITS/pretrained_models/s1v3.ckpt
    GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large
    GPT_SoVITS/pretrained_models/chinese-hubert-base
Traceback (most recent call last):
  File "/content/GPT-SoVITS/GPT_SoVITS/download.py", line 6, in <module>
    from text.g2pw import G2PWPinyin
  File "/content/GPT-SoVITS/GPT_SoVITS/text/g2pw/__init__.py", line 1, in <module>
    from text.g2pw.g2pw import *
  File "/content/GPT-SoVITS/GPT_SoVITS/text/g2pw/g2pw.py", line 12, in <module>
    from .onnx_api import G2PWOnnxConverter
  File "/content/GPT-SoVITS/GPT_SoVITS/text/g2pw/onnx_api.py", line 11, in <module>
    import onnxruntime
  File "/usr/local/lib/python3.12/dist-packages/onnxruntime/__init__.py", line 78, in <module>
    rais

In [2]:
# 1. 修复 onnxruntime 和 CUDA 版本冲突（换成 CPU 版，这部分模型转换用不用 GPU 都行，速度差异很小）
!pip uninstall -y onnxruntime-gpu
!pip install onnxruntime

# 2. 修复 webui.py 缺少 share=True 的问题
!sed -i '/app.queue().launch(/a\    share=True,' /content/GPT-SoVITS/webui.py

# 3. 重新下载缺失的预训练模型（现在 onnxruntime 问题修好了，应该能顺利跑完）
%cd /content/GPT-SoVITS
!python GPT_SoVITS/download.py

# 4. 启动 webui
!python webui.py

Found existing installation: onnxruntime-gpu 1.27.0
Uninstalling onnxruntime-gpu-1.27.0:
  Successfully uninstalled onnxruntime-gpu-1.27.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 94.1 MB/s eta 0:00:00
/content/GPT-SoVITS
Rebuilding Polyphonic Dictionary: 13b2211c317c75794123ffdf7c2aea021c75b0c606ad61d7c8b05bb00b64fa21 -> e5112f5800e8b41a4a94be20c5c18bf1bda17950d87be0788f712e0bf1b9ed36
Extracting g2pw model...
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py", line 424, in cached_files
    hf_hub_download(
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 106, in _inner_fn
    validate_repo_id(arg_value)
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 154, in validate_repo_id
    raise HFValidationError(
huggingface_hub.errors.HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': 'GPT_SoVITS/pret

In [ ]:
# 1. 撤销上次误加的重复参数
!sed -i '/^    share=True,$/d' /content/GPT-SoVITS/webui.py

# 2. 直接把全部官方预训练模型一次性下载到正确目录（比之前调用 download.py 更稳）
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="lj1995/GPT-SoVITS",
    local_dir="/content/GPT-SoVITS/GPT_SoVITS/pretrained_models",
)

# 3. 用官方推荐的正确方式开启公网链接（改环境变量，不碰源码）
import os
os.environ["is_share"] = "True"

%cd /content/GPT-SoVITS
!python webui.py

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 27 files:   0%|          | 0/27 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

README.md:   0%|          | 0.00/114 [00:00<?, ?B/s]

...

v2Pro/s2Dv2Pro.pth:   0%|          | 0.00/126M [00:00<?, ?B/s]

v2Pro/s2Dv2ProPlus.pth:   0%|          | 0.00/126M [00:00<?, ?B/s]

v2Pro/s2Gv2Pro.pth:   0%|          | 0.00/162M [00:00<?, ?B/s]

v2Pro/s2Gv2ProPlus.pth:   0%|          | 0.00/200M [00:00<?, ?B/s]

/content/GPT-SoVITS
Running on local URL:  http://0.0.0.0:9874
ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 62, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 90, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr

In [ ]:
!pip install "starlette<1.0.0" --force-reinstall --no-deps -q
%cd /content/GPT-SoVITS
!python webui.py